# Module 3b: final scoring (test split and own recordings, ONCE)
Run only after `gate.threshold` is set and committed, and after `python -m readback_guard.batch test` and `... batch own`.

In [ ]:
import subprocess
import pandas as pd
from readback_guard.config import ROOT, load_config
from readback_guard.asr import load_asr_cases, load_own_cases, wer_table, critical_values
from readback_guard.evaluate import load_cases, rates, log_result, attribution, mcnemar, md_table
from readback_guard.score import run_cases

cfg = load_config()
t = cfg["gate"]["threshold"]
assert t is not None, "set gate.threshold on dev first"
dirty = subprocess.run(["git", "status", "--porcelain", "labels", "configs"], cwd=ROOT, capture_output=True, text=True).stdout
assert not dirty, "commit labels/ and configs/ before scoring"
t

In [ ]:
test = load_cases(cfg, "test")
test_real = test.query("kind == 'real'")
test_asr = load_asr_cases(cfg, "test")

runs = {
    "1b human text, real + injected": run_cases(test, cfg),
    "human text, real pairs": run_cases(test_real, cfg),
    "Whisper text, real pairs": run_cases(test_asr, cfg),
    "Whisper + gate, real pairs": run_cases(test_asr, cfg, t),
}
main = pd.DataFrame({name: rates(df) for name, df in runs.items()}).T
main

In [ ]:
for name, df in runs.items():
    log_result("final_" + name.split(",")[0].replace(" ", "_").replace("+", "plus"), "test",
               "asr" if "Whisper" in name else "ref", rates(df), cfg, notes=name)

In [ ]:
wer = wer_table(test_asr)
cv = critical_values(test_asr)
print("WER", wer, " critical-value error rate", round(1 - cv.asr_ok.mean(), 3), "of", len(cv))
cv.groupby(["side", "type"]).asr_ok.agg(["size", "mean"]).round(2)

In [ ]:
m, table = attribution(runs["human text, real pairs"], runs["Whisper text, real pairs"])
mg, table_g = attribution(runs["human text, real pairs"], runs["Whisper + gate, real pairs"])
print("human vs Whisper:", mcnemar(m))
print("human vs Whisper + gate:", mcnemar(mg))
table

## Own recordings (real speech with real errors)

In [ ]:
own_human = run_cases(load_own_cases(cfg, "human"), cfg)
own_asr_cases = load_own_cases(cfg, "asr")
own_asr = run_cases(own_asr_cases, cfg)
own_gate = run_cases(own_asr_cases, cfg, t)
own = pd.DataFrame({"script text": rates(own_human), "Whisper text": rates(own_asr), "Whisper + gate": rates(own_gate)}).T
for name, df in [("own_script", own_human), ("own_whisper", own_asr), ("own_whisper_gate", own_gate)]:
    log_result(name, "own", "asr" if "whisper" in name else "ref", rates(df), cfg)
own

In [ ]:
mo, table_o = attribution(own_human, own_asr)
print(mcnemar(mo))
per_rec = own_human[["case_id", "verdict", "pred"]].rename(columns={"pred": "script"})
per_rec["whisper"], per_rec["whisper_gate"], per_rec["reason"] = own_asr.pred, own_gate.pred, own_gate.reason
per_rec

In [ ]:
cols = ["n_errors", "detected", "detection_rate", "detection_ci", "n_correct", "false_alarms",
        "false_alarm_rate", "false_alarm_ci", "false_accusations", "missed", "say_again", "say_again_rate"]
parts = ["## Test split", md_table(main[cols]), f"WER {wer}, critical-value error rate {1 - cv.asr_ok.mean():.3f} (n={len(cv)})",
         "## Attribution, human vs Whisper (test, real pairs)", md_table(table), str(mcnemar(m)),
         "## Attribution, human vs Whisper + gate", md_table(table_g), str(mcnemar(mg)),
         "## Own recordings", md_table(own[cols]), md_table(per_rec.set_index("case_id"))]
(ROOT / "results" / "tables.md").write_text("\n\n".join(parts))
print("wrote results/tables.md")